# 8. Planning and Learning with Tabular Methods

**Kaynak:** Sutton & Barto, *Reinforcement Learning: An Introduction*, 2nd Edition (2018)
- **Bölüm 8: Planning and Learning with Tabular Methods** (Sayfa 163-188)

## İçindekiler
1. Models and Planning *(s. 163-166)*
2. Dyna Architecture *(s. 166-168)*
3. Dyna-Q Algorithm *(s. 168-170)*
4. When the Model Is Wrong *(s. 170-174)*
5. Prioritized Sweeping *(s. 174-176)*
6. Planning at Decision Time *(s. 180-182)*

---
## 8.1 Model-based vs Model-free

📖 **Referans:** Sutton & Barto, Sayfa 163-166, Section 8.1-8.2

> *"By a model of the environment we mean anything that an agent can use to predict how the environment will respond to its actions."* (s. 163)

### Model-free Learning
- MC, TD, SARSA, Q-Learning
- Sadece deneyimden öğren
- Model yok

### Model-based Learning
- Environment'ın bir **modelini** öğren veya kullan
- Model üzerinde **simülasyon** yap (planning)
- > *"More sample efficient... because they can use experience to more than just improve a value function and a policy"* (s. 166)

### Model Nedir? (s. 163)

Model, agent'ın environment hakkındaki bilgisidir:
- **Distribution model**: $p(s', r | s, a)$ - tam olasılık dağılımı
- **Sample model**: $(s, a) \rightarrow s', r$ - sample üretir

> *"Distribution models are stronger than sample models in that they can always be used to produce samples"* (s. 163)

In [ ]:
# Kod Örneği: Maze Environment
# Referans: Example 8.1 (s. 168) - Simple Maze

import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
import heapq

class MazeEnv:
    """
    Maze environment for Dyna experiments.
    Referans: Example 8.1 (s. 168)
    
    "Simple maze... the upper left of each subtable shows the situation 
    after just one episode"
    """
    
    def __init__(self, maze=None):
        if maze is None:
            # Default maze (Example 8.1, Figure 8.3)
            self.maze = np.array([
                [0, 0, 0, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 0, 0, 0, 0, 0],
                [1, 1, 1, 1, 1, 1, 1, 1, 0],  # Wall
                [0, 0, 0, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 0, 0, 0, 0, 0]
            ])  # 1 = wall (obstacle), 0 = free
        else:
            self.maze = maze
        
        self.rows, self.cols = self.maze.shape
        self.start = (5, 3)  # S in Figure 8.3
        self.goal = (0, 8)   # G in Figure 8.3
        self.n_actions = 4
        
        self.actions = {
            0: (-1, 0),  # up
            1: (0, 1),   # right
            2: (1, 0),   # down
            3: (0, -1)   # left
        }
        
        self.reset()
    
    def reset(self):
        self.position = self.start
        return self.position
    
    def step(self, action):
        """
        "The reward is zero on all transitions except those on which 
        the goal is reached, when it is +1" (s. 168)
        """
        row, col = self.position
        drow, dcol = self.actions[action]
        
        new_row = row + drow
        new_col = col + dcol
        
        # "Actions that would take the agent into an obstacle or 
        # outside the grid leave the position unchanged" (s. 168)
        if (0 <= new_row < self.rows and 
            0 <= new_col < self.cols and
            self.maze[new_row, new_col] == 0):
            self.position = (new_row, new_col)
        
        if self.position == self.goal:
            return self.position, 1, True
        
        return self.position, 0, False

def visualize_maze(env, Q=None, path=None, title="Maze"):
    """Maze'i görselleştir - Figure 8.3 (s. 168)"""
    fig, ax = plt.subplots(figsize=(10, 7))
    
    for row in range(env.rows):
        for col in range(env.cols):
            if env.maze[row, col] == 1:
                color = 'black'
            elif (row, col) == env.start:
                color = 'lightgreen'
            elif (row, col) == env.goal:
                color = 'gold'
            else:
                color = 'white'
            
            rect = plt.Rectangle((col, env.rows - 1 - row), 1, 1,
                                 facecolor=color, edgecolor='gray', linewidth=0.5)
            ax.add_patch(rect)
    
    if path:
        path_x = [p[1] + 0.5 for p in path]
        path_y = [env.rows - p[0] - 0.5 for p in path]
        ax.plot(path_x, path_y, 'b-o', linewidth=2, markersize=4)
    
    ax.set_xlim(0, env.cols)
    ax.set_ylim(0, env.rows)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title(title)
    plt.show()

env = MazeEnv()
visualize_maze(env, title="Maze Environment - Figure 8.3 (s. 168)")

---
## 8.2 Dyna Architecture

📖 **Referans:** Sutton & Barto, Sayfa 166-168, Section 8.2

> *"The Dyna-Q architecture... integrates the major functions needed in an online planning agent."* (s. 166)

**Dyna**, planning ve learning'i birleştirir - Figure 8.2 (s. 167):

1. **Direct RL**: Gerçek deneyimden öğren
2. **Model Learning**: Deneyimden model öğren  
3. **Planning**: Model kullanarak simüle edilmiş deneyimden öğren

> *"In Dyna-Q, learning and planning are accomplished by exactly the same algorithm, operating on real experience for learning and on simulated experience for planning."* (s. 166)

```
Real Experience → Direct RL → Q/V
      ↓
   Model
      ↓
Simulated Experience → Planning → Q/V
```

---
## 8.3 Dyna-Q Algorithm

📖 **Referans:** Sutton & Barto, Sayfa 168-170, Algorithm on page 169

> *"Within a planning agent, real experience is used for two purposes: first, to improve the model...; second, to directly improve the value function and policy."* (s. 168)

### Tabular Dyna-Q Algorithm (s. 169)

```
Initialize Q(s, a) and Model(s, a) for all s and a
Loop forever:
    (a) S ← current (nonterminal) state
    (b) A ← ε-greedy(S, Q)
    (c) Take action A; observe resultant reward, R, and state, S'
    (d) Q(S, A) ← Q(S, A) + α[R + γ max_a Q(S', a) - Q(S, A)]  # Direct RL
    (e) Model(S, A) ← R, S' (assuming deterministic environment)  # Model Learning
    (f) Loop repeat n times:  # Planning
        S ← random previously observed state
        A ← random action previously taken in S
        R, S' ← Model(S, A)
        Q(S, A) ← Q(S, A) + α[R + γ max_a Q(S', a) - Q(S, A)]
```

In [ ]:
class DynaQ:
    """
    Dyna-Q Algorithm.
    Referans: Algorithm (s. 169) - "Tabular Dyna-Q"
    """
    
    def __init__(self, env, alpha=0.1, gamma=0.95, epsilon=0.1, n_planning=5):
        self.env = env
        self.alpha = alpha  # Step size
        self.gamma = gamma  # Discount factor
        self.epsilon = epsilon
        self.n_planning = n_planning  # "n" planning steps
        
        # "Initialize Q(s, a) and Model(s, a)"
        self.Q = defaultdict(lambda: np.zeros(env.n_actions))
        self.model = {}  # Model(s, a) = (r, s')
        self.visited_sa = []  # List of visited (s, a) pairs
    
    def select_action(self, state):
        """(b) A ← ε-greedy(S, Q)"""
        if np.random.random() < self.epsilon:
            return np.random.randint(self.env.n_actions)
        return np.argmax(self.Q[state])
    
    def learn(self, state, action, reward, next_state):
        """(d) Q(S,A) ← Q(S,A) + α[R + γ max_a Q(S',a) - Q(S,A)]"""
        td_target = reward + self.gamma * np.max(self.Q[next_state])
        self.Q[state][action] += self.alpha * (td_target - self.Q[state][action])
    
    def update_model(self, state, action, reward, next_state):
        """(e) Model(S,A) ← R, S' (assuming deterministic environment)"""
        if (state, action) not in self.model:
            self.visited_sa.append((state, action))
        self.model[(state, action)] = (reward, next_state)
    
    def planning(self):
        """
        (f) Loop repeat n times - Planning steps
        "selecting random previously observed states and actions"
        """
        if not self.visited_sa:
            return
        
        for _ in range(self.n_planning):
            # "S ← random previously observed state"
            # "A ← random action previously taken in S"
            idx = np.random.randint(len(self.visited_sa))
            state, action = self.visited_sa[idx]
            
            # "R, S' ← Model(S, A)"
            reward, next_state = self.model[(state, action)]
            
            # Same Q-learning update
            self.learn(state, action, reward, next_state)
    
    def run_episode(self):
        """Run one episode."""
        state = self.env.reset()  # (a) S ← current state
        steps = 0
        
        while True:
            action = self.select_action(state)  # (b)
            next_state, reward, done = self.env.step(action)  # (c)
            
            # (d) Direct RL
            self.learn(state, action, reward, next_state)
            
            # (e) Model learning
            self.update_model(state, action, reward, next_state)
            
            # (f) Planning
            self.planning()
            
            steps += 1
            
            if done:
                break
            
            state = next_state
        
        return steps

In [ ]:
# Figure 8.4 (s. 170) - Effect of planning steps on learning
# "with n = 0 no planning at all... With n = 5, performance is much improved...
# n = 50, the agent is able to find a much shorter path"

def run_dyna_experiment(env, n_planning_values, n_episodes=50, n_runs=10):
    """
    Compare Dyna-Q with different planning steps.
    Referans: Figure 8.4 (s. 170)
    """
    
    results = {}
    
    for n_planning in n_planning_values:
        all_steps = np.zeros((n_runs, n_episodes))
        
        for run in range(n_runs):
            agent = DynaQ(env, n_planning=n_planning)
            
            for ep in range(n_episodes):
                steps = agent.run_episode()
                all_steps[run, ep] = steps
        
        results[n_planning] = all_steps.mean(axis=0)
    
    return results

n_planning_values = [0, 5, 50]
results = run_dyna_experiment(env, n_planning_values, n_episodes=50, n_runs=20)

# Plot - Figure 8.4 (s. 170)
plt.figure(figsize=(10, 5))

for n, steps in results.items():
    plt.plot(steps, label=f'n={n} planning steps', linewidth=2)

plt.xlabel('Episodes')
plt.ylabel('Steps per Episode')
plt.title('Dyna-Q: Effect of Planning Steps - Figure 8.4 (s. 170)')
plt.legend()
plt.yscale('log')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Train and visualize learned policy
agent = DynaQ(env, n_planning=50)

for _ in range(50):
    agent.run_episode()

# Extract path using greedy policy
def get_greedy_path(env, Q, max_steps=100):
    path = []
    state = env.reset()
    path.append(state)
    
    for _ in range(max_steps):
        action = np.argmax(Q[state])
        next_state, _, done = env.step(action)
        path.append(next_state)
        
        if done:
            break
        state = next_state
    
    return path

path = get_greedy_path(env, agent.Q)
visualize_maze(env, path=path, title=f"Dyna-Q Learned Path ({len(path)-1} steps)")

---
## 8.4 When the Model Is Wrong

📖 **Referans:** Sutton & Barto, Sayfa 170-174, Section 8.3

> *"Models may be incorrect because the environment is stochastic and only a limited number of samples have been observed, or because the model was learned using function approximation... or simply because the environment has changed"* (s. 170)

Model-based learning'in bir zorluğu: **environment değişirse** model yanılır.

### Dyna-Q+ (s. 172-173)

> *"The general idea is to encourage behavior that tests long-untried actions."* (s. 172)

Uzun süredir denenmemiş state-action çiftlerine **exploration bonus** ekle:

$$r + \kappa \sqrt{\tau(s, a)}$$

> *"where τ(s, a) is the number of time steps since (s, a) was last tried... and κ is a small constant"* (s. 172)

Bu bonus, "stale" bilgiyi test etmeyi teşvik eder.

In [ ]:
class DynaQPlus:
    """
    Dyna-Q+ with exploration bonus.
    Referans: "Dyna-Q+ agent" (s. 172-173)
    
    "The Dyna-Q+ agent uses such a bonus... it adds κ√τ 
    to the reward in the planning updates"
    """
    
    def __init__(self, env, alpha=0.1, gamma=0.95, epsilon=0.1, 
                 n_planning=5, kappa=0.001):
        self.env = env
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        self.n_planning = n_planning
        self.kappa = kappa  # "κ is a small constant" (s. 172)
        
        self.Q = defaultdict(lambda: np.zeros(env.n_actions))
        self.model = {}
        self.time_since_visit = defaultdict(lambda: 0)  # "τ(s, a)"
        self.time_step = 0
        
        self.visited_states = set()
    
    def select_action(self, state):
        if np.random.random() < self.epsilon:
            return np.random.randint(self.env.n_actions)
        return np.argmax(self.Q[state])
    
    def learn(self, state, action, reward, next_state):
        td_target = reward + self.gamma * np.max(self.Q[next_state])
        self.Q[state][action] += self.alpha * (td_target - self.Q[state][action])
    
    def update_model(self, state, action, reward, next_state):
        self.model[(state, action)] = (reward, next_state)
        self.visited_states.add(state)
        self.time_since_visit[(state, action)] = self.time_step
    
    def planning(self):
        if not self.visited_states:
            return
        
        visited_list = list(self.visited_states)
        
        for _ in range(self.n_planning):
            state = visited_list[np.random.randint(len(visited_list))]
            action = np.random.randint(self.env.n_actions)
            
            # "Actions never before tried are assumed to transition 
            # back to the same state with reward of zero" (s. 172)
            if (state, action) in self.model:
                reward, next_state = self.model[(state, action)]
            else:
                reward, next_state = 0, state
            
            # "κ√τ" exploration bonus
            tau = self.time_step - self.time_since_visit[(state, action)]
            bonus = self.kappa * np.sqrt(tau)
            
            # Update with bonus: "adds κ√τ to the reward"
            td_target = (reward + bonus) + self.gamma * np.max(self.Q[next_state])
            self.Q[state][action] += self.alpha * (td_target - self.Q[state][action])
    
    def run_episode(self):
        state = self.env.reset()
        steps = 0
        
        while True:
            self.time_step += 1
            action = self.select_action(state)
            next_state, reward, done = self.env.step(action)
            
            self.learn(state, action, reward, next_state)
            self.update_model(state, action, reward, next_state)
            self.planning()
            
            steps += 1
            
            if done:
                break
            state = next_state
        
        return steps

In [ ]:
# Changing maze experiment
class ChangingMazeEnv(MazeEnv):
    """Maze that changes at a certain time step."""
    
    def __init__(self):
        # Initial maze (longer path required)
        maze = np.array([
            [0, 0, 0, 0, 0, 0, 0, 0, 0],
            [0, 0, 0, 0, 0, 0, 0, 0, 0],
            [0, 0, 0, 0, 0, 0, 0, 0, 0],
            [1, 1, 1, 1, 1, 1, 1, 1, 0],
            [0, 0, 0, 0, 0, 0, 0, 0, 0],
            [0, 0, 0, 0, 0, 0, 0, 0, 0]
        ])
        super().__init__(maze)
        self.changed = False
    
    def change_maze(self):
        """Open a shortcut."""
        self.maze[3, 0] = 0  # Remove wall block
        self.changed = True

# Compare Dyna-Q and Dyna-Q+ on changing maze
def run_changing_maze_experiment(n_episodes=200, change_at=100, n_runs=10):
    dyna_q_rewards = np.zeros((n_runs, n_episodes))
    dyna_q_plus_rewards = np.zeros((n_runs, n_episodes))
    
    for run in range(n_runs):
        # Dyna-Q
        env1 = ChangingMazeEnv()
        agent1 = DynaQ(env1, n_planning=10)
        
        # Dyna-Q+
        env2 = ChangingMazeEnv()
        agent2 = DynaQPlus(env2, n_planning=10, kappa=0.001)
        
        for ep in range(n_episodes):
            if ep == change_at:
                env1.change_maze()
                env2.change_maze()
            
            steps1 = agent1.run_episode()
            steps2 = agent2.run_episode()
            
            dyna_q_rewards[run, ep] = 1 if steps1 < 50 else 0
            dyna_q_plus_rewards[run, ep] = 1 if steps2 < 50 else 0
    
    return dyna_q_rewards.mean(axis=0), dyna_q_plus_rewards.mean(axis=0)

print("Running changing maze experiment...")
dyna_q_perf, dyna_q_plus_perf = run_changing_maze_experiment(n_runs=5)

# Cumulative rewards plot
plt.figure(figsize=(10, 5))
plt.plot(np.cumsum(dyna_q_perf), label='Dyna-Q', linewidth=2)
plt.plot(np.cumsum(dyna_q_plus_perf), label='Dyna-Q+', linewidth=2)
plt.axvline(x=100, color='gray', linestyle='--', label='Maze changes')
plt.xlabel('Episode')
plt.ylabel('Cumulative Reward')
plt.title('Dyna-Q vs Dyna-Q+ on Changing Maze')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

---
## 8.5 Prioritized Sweeping

📖 **Referans:** Sutton & Barto, Sayfa 174-176, Section 8.4

> *"In the Dyna agents presented in the preceding sections, simulated transitions are started in state–action pairs selected uniformly at random from all previously experienced pairs. But a uniform selection is usually not the best"* (s. 174)

Random planning yerine, **öncelikli** güncelleme yap:

**Fikir**: TD error'u büyük olan state'leri önce güncelle.

> *"The agent maintains a queue of every state–action pair whose estimated value would change nontrivially if updated, prioritized by the size of the change."* (s. 174)

### Algoritma (s. 175)
1. Gerçek deneyimden öğren
2. TD error hesapla: $|R + \gamma \max_a Q(S', a) - Q(S, A)|$
3. Error threshold'u aşarsa, (S, A)'yı priority queue'ya ekle
4. Planning: Queue'dan en yüksek öncelikli (s, a)'yı al ve güncelle
5. Bu güncelleme, **predecessor** state'lerin error'unu artırabilir → onları da queue'ya ekle

In [ ]:
class PrioritizedSweeping:
    """
    Prioritized Sweeping Algorithm.
    Referans: Algorithm (s. 175) - "Prioritized Sweeping"
    """
    
    def __init__(self, env, alpha=0.1, gamma=0.95, epsilon=0.1, 
                 n_planning=5, theta=0.0001):
        self.env = env
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        self.n_planning = n_planning
        self.theta = theta  # "θ: threshold on P for insertion into PQueue" (s. 175)
        
        self.Q = defaultdict(lambda: np.zeros(env.n_actions))
        self.model = {}
        # "model(s, a) is the predicted next state and reward"
        self.predecessors = defaultdict(set)  # "Model(s, a) includes S̄ → (S, A)"
        self.priority_queue = []  # "PQueue: a priority queue"
        self.in_queue = set()
    
    def select_action(self, state):
        if np.random.random() < self.epsilon:
            return np.random.randint(self.env.n_actions)
        return np.argmax(self.Q[state])
    
    def get_priority(self, state, action):
        """Calculate priority: "P ← |R + γ max_a Q(S̄, a) − Q(S, A)|" """
        if (state, action) not in self.model:
            return 0
        reward, next_state = self.model[(state, action)]
        td_error = abs(reward + self.gamma * np.max(self.Q[next_state]) - self.Q[state][action])
        return td_error
    
    def add_to_queue(self, state, action, priority):
        """Add (s, a) to priority queue if priority > theta."""
        if priority > self.theta:  # "If P > θ"
            if (state, action) not in self.in_queue:
                # "insert S, A into PQueue with priority P"
                heapq.heappush(self.priority_queue, (-priority, state, action))
                self.in_queue.add((state, action))
    
    def update_model(self, state, action, reward, next_state):
        """Update model and predecessors."""
        self.model[(state, action)] = (reward, next_state)
        # "Model(s, a) ← R, S̄ (and S̄ → (S, A))"
        self.predecessors[next_state].add((state, action))
    
    def planning(self):
        """
        "Loop repeat n times, while PQueue is not empty"
        """
        for _ in range(self.n_planning):
            if not self.priority_queue:
                break
            
            # "S, A ← first(PQueue)"
            _, state, action = heapq.heappop(self.priority_queue)
            self.in_queue.discard((state, action))
            
            # Update Q: "Q(S, A) ← Q(S, A) + α[R + γ max_a Q(S̄, a) − Q(S, A)]"
            reward, next_state = self.model[(state, action)]
            td_target = reward + self.gamma * np.max(self.Q[next_state])
            self.Q[state][action] += self.alpha * (td_target - self.Q[state][action])
            
            # "Loop for all S̄, Ā predicted to lead to S:"
            for pred_state, pred_action in self.predecessors[state]:
                priority = self.get_priority(pred_state, pred_action)
                self.add_to_queue(pred_state, pred_action, priority)
    
    def run_episode(self):
        state = self.env.reset()
        steps = 0
        
        while True:
            action = self.select_action(state)
            next_state, reward, done = self.env.step(action)
            
            self.update_model(state, action, reward, next_state)
            
            priority = self.get_priority(state, action)
            self.add_to_queue(state, action, priority)
            
            self.planning()
            
            steps += 1
            
            if done:
                break
            state = next_state
        
        return steps

In [ ]:
# Compare Dyna-Q and Prioritized Sweeping
def compare_planning_methods(env, n_episodes=30, n_runs=10):
    dyna_steps = np.zeros((n_runs, n_episodes))
    ps_steps = np.zeros((n_runs, n_episodes))
    
    for run in range(n_runs):
        agent_dyna = DynaQ(env, n_planning=5)
        agent_ps = PrioritizedSweeping(env, n_planning=5)
        
        for ep in range(n_episodes):
            dyna_steps[run, ep] = agent_dyna.run_episode()
            ps_steps[run, ep] = agent_ps.run_episode()
    
    return dyna_steps.mean(axis=0), ps_steps.mean(axis=0)

dyna_avg, ps_avg = compare_planning_methods(env, n_episodes=30, n_runs=10)

plt.figure(figsize=(10, 5))
plt.plot(dyna_avg, label='Dyna-Q', linewidth=2)
plt.plot(ps_avg, label='Prioritized Sweeping', linewidth=2)
plt.xlabel('Episode')
plt.ylabel('Steps per Episode')
plt.title('Dyna-Q vs Prioritized Sweeping')
plt.legend()
plt.yscale('log')
plt.grid(True, alpha=0.3)
plt.show()

---
## 8.6 Planning at Decision Time

📖 **Referans:** Sutton & Barto, Sayfa 180-182, Section 8.8

> *"Planning can be done in two fundamentally different ways."* (s. 180)

### Background Planning (s. 180)
- Her adımda model ile simülasyon
- Q/V'yi sürekli iyileştir
- Dyna-Q, Prioritized Sweeping

> *"The planning is ongoing and is not focused on the current state"*

### Decision-Time Planning (s. 180)
- Sadece karar anında plan yap
- Mevcut state'ten başlayarak ileriye bak
- Örnek: **Monte Carlo Tree Search (MCTS)**

> *"Planning begins and ends with the current state"*

### Rollout Algorithms (s. 181)

> *"Rollout algorithms are decision-time planning algorithms based on Monte Carlo control applied to simulated trajectories"*

Basit bir decision-time planning:
1. Mevcut state'ten başla
2. Base policy ile simülasyon yap (rollout)
3. Return'ü hesapla
4. En iyi return veren aksiyonu seç

In [ ]:
class RolloutAgent:
    """
    Simple Rollout Algorithm for decision-time planning.
    Referans: "Rollout algorithms" (s. 181)
    
    "The goal of a rollout algorithm is to improve upon the rollout policy"
    """
    
    def __init__(self, env, n_rollouts=10, max_depth=50, gamma=0.95):
        self.env = env
        self.n_rollouts = n_rollouts
        self.max_depth = max_depth
        self.gamma = gamma
    
    def base_policy(self, state):
        """
        "rollout policy" - Random base policy for rollouts.
        "typically a simple, heuristic policy without optimality guarantees"
        """
        return np.random.randint(self.env.n_actions)
    
    def rollout(self, state, first_action):
        """
        Perform one rollout from state with first_action.
        "simulated trajectories that all begin at the current environment state"
        """
        original_pos = self.env.position
        
        self.env.position = state
        next_state, reward, done = self.env.step(first_action)
        
        total_return = reward
        discount = self.gamma
        
        for _ in range(self.max_depth):
            if done:
                break
            
            action = self.base_policy(next_state)
            next_state, reward, done = self.env.step(action)
            total_return += discount * reward
            discount *= self.gamma
        
        self.env.position = original_pos
        
        return total_return
    
    def select_action(self, state):
        """
        Select action using rollouts.
        "For each possible action from the current state, the algorithm 
        performs many simulated trajectories"
        """
        action_values = np.zeros(self.env.n_actions)
        
        for action in range(self.env.n_actions):
            returns = []
            for _ in range(self.n_rollouts):
                ret = self.rollout(state, action)
                returns.append(ret)
            action_values[action] = np.mean(returns)
        
        return np.argmax(action_values)

# Test rollout agent
rollout_agent = RolloutAgent(env, n_rollouts=20)

state = env.reset()
path = [state]
total_steps = 0

for _ in range(100):
    action = rollout_agent.select_action(env.position)
    next_state, reward, done = env.step(action)
    path.append(next_state)
    total_steps += 1
    
    if done:
        break

visualize_maze(env, path=path, title=f"Rollout Agent Path ({total_steps} steps)")

---
## Özet

📖 **Chapter 8 Key Points (s. 163-188)**

| Yöntem | Sayfa | Tip | Avantaj | Dezavantaj |
|--------|-------|-----|---------|------------|
| **Model-free** | s. 163 | - | Basit, model gereksiz | Sample inefficient |
| **Dyna-Q** | s. 168-170 | Background | Sample efficient | Yanlış model tehlikesi |
| **Dyna-Q+** | s. 172-173 | Background | Exploration bonus | Daha karmaşık |
| **Prioritized Sweeping** | s. 174-176 | Background | Daha hızlı yakınsama | Predecessor tracking |
| **Rollout** | s. 181 | Decision-time | Basit, flexible | Hesaplama maliyeti |

### Önemli Kavramlar

> *"All state-space planning methods involve computing value functions as a key intermediate step toward improving the policy"* (s. 165)

- **Planning**: Model kullanarak value/policy iyileştirme
- **Model Learning**: Deneyimden model öğrenme
- **Sample Efficiency**: Daha az gerçek deneyimle öğrenme

### Anahtar Denklem

**Dyna-Q Planning Update** (s. 169):
$$Q(S, A) \leftarrow Q(S, A) + \alpha [R + \gamma \max_a Q(S', a) - Q(S, A)]$$

---
### Sonraki Notebook
**09 - Policy Gradient Methods** *(Chapter 13, s. 327-352)*: Direct policy optimization